In [ ]:
from pathlib import Path
import json
import csv
from collections import Counter
from collections import defaultdict
import pandas as pd


file_path = Path("../../data/raw/entities.ftm.json")

schema_counts = Counter()

with file_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue

        obj = json.loads(line)
        schema_counts[obj.get("schema")] += 1

        if i >= 9999:   # first 10,000 lines
            break

print(schema_counts.most_common(20))

[('Person', 1109262), ('Occupancy', 851502), ('Sanction', 612182), ('Security', 354075), ('Company', 234693), ('Ownership', 227274), ('Position', 174349), ('Directorship', 131859), ('Organization', 101660), ('LegalEntity', 84282), ('Family', 74232), ('UnknownLink', 41943), ('Succession', 36365), ('Address', 30868), ('CryptoWallet', 13695), ('Documentation', 13114), ('Passport', 11091), ('Vessel', 9021), ('Identification', 6777), ('Article', 3460)]


In [ ]:
# -------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------
input_file = Path("../../data/raw/entities.ftm.json")
output_dir = Path("../../data/processed/schema_csvs")
output_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------------
def safe_value(value):
    """
    Preserve structure safely for CSV:
    - lists/dicts -> JSON string
    - everything else stays as is
    """
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value

def flatten_entity(entity):
    """
    Flatten one entity safely:
    - top-level keys stay as they are
    - properties keys are prefixed with 'properties.'
    """
    flat = {}

    for key, value in entity.items():
        if key != "properties":
            flat[key] = safe_value(value)

    for key, value in entity.get("properties", {}).items():
        flat[f"properties.{key}"] = safe_value(value)

    return flat

# -------------------------------------------------------------------
# FIRST PASS:
# 1. collect rows by schema
# 2. collect all columns seen for each schema
# -------------------------------------------------------------------
rows_by_schema = defaultdict(list)
columns_by_schema = defaultdict(set)

with input_file.open("r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            entity = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Skipping bad JSON on line {line_num}: {e}")
            continue

        schema = entity.get("schema", "UNKNOWN")
        flat = flatten_entity(entity)

        rows_by_schema[schema].append(flat)
        columns_by_schema[schema].update(flat.keys())

# -------------------------------------------------------------------
# SECOND PASS:
# write one CSV per schema
# -------------------------------------------------------------------
for schema, rows in rows_by_schema.items():
    columns = sorted(columns_by_schema[schema])
    output_file = output_dir / f"{schema}.csv"

    with output_file.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()

        for row in rows:
            complete_row = {col: row.get(col, "") for col in columns}
            writer.writerow(complete_row)

    print(f"Saved: {output_file} ({len(rows)} rows, {len(columns)} columns)")

In [ ]:
# -------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------
file_path = Path("../../data/processed/schema_csvs/Person.csv")

df = pd.read_csv(file_path)

# -------------------------------------------------------------------
# BASIC OVERVIEW
# -------------------------------------------------------------------
print("Shape (rows, columns):", df.shape)
print("\nColumns:\n", df.columns.tolist())

# -------------------------------------------------------------------
# SAMPLE DATA
# -------------------------------------------------------------------
print("\nFirst 5 rows:")
print(df.head())

# -------------------------------------------------------------------
# DATA TYPES
# -------------------------------------------------------------------
print("\nData types:")
print(df.dtypes)

# -------------------------------------------------------------------
# MISSING VALUES
# -------------------------------------------------------------------
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

# -------------------------------------------------------------------
# NON-NULL COUNTS (useful!)
# -------------------------------------------------------------------
print("\nNon-null counts:")
print(df.count().sort_values(ascending=False))

# -------------------------------------------------------------------
# UNIQUE VALUES (for key columns)
# -------------------------------------------------------------------
print("\nUnique schemas:", df["schema"].unique() if "schema" in df.columns else "N/A")

if "properties.country" in df.columns:
    print("\nUnique countries (sample):")
    print(df["properties.country"].dropna().unique()[:10])

if "properties.topics" in df.columns:
    print("\nUnique topics (sample):")
    print(df["properties.topics"].dropna().unique()[:10])

# -------------------------------------------------------------------
# BASIC STATS (for numeric columns if any)
# -------------------------------------------------------------------
print("\nSummary statistics:")
print(df.describe(include="all"))

# -------------------------------------------------------------------
# COLUMN INSPECTION (VERY IMPORTANT)
# -------------------------------------------------------------------
# pick a few important columns to inspect deeply

cols_to_check = [
    "properties.name",
    "properties.country",
    "properties.birthDate",
    "properties.topics"
]

for col in cols_to_check:
    if col in df.columns:
        print(f"\nSample values for {col}:")
        print(df[col].dropna().head(5))